In [2]:
from docling.document_converter import DocumentConverter


c:\Users\mateus.thomasini\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
source = "https://ged-anchieta.s3.amazonaws.com/GED/Documentos/636341Media-5-45333719-17c8-44a0-863a-45d7868c51e51506881527285212-mesclado.pdf"  # file path or URL
converter = DocumentConverter()
doc = converter.convert(source)

2026-01-20 07:18:01,590 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-20 07:18:01,597 - INFO - Going to convert document batch...
2026-01-20 07:18:01,598 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-01-20 07:18:01,600 - INFO - rapidocr cannot be used because onnxruntime is not installed.
2026-01-20 07:18:01,601 - INFO - easyocr cannot be used because it is not installed.
2026-01-20 07:18:01,601 - INFO - Accelerator device: 'cpu'
[INFO] 2026-01-20 07:18:01,623 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-20 07:18:01,624 [RapidOCR] device_config.py:50: Using CPU device
[INFO] 2026-01-20 07:18:01,638 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\mateus.thomasini\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-20 07:18:01,640 [RapidOCR] main.py:50: Using C:\Users\mateus.thomasini\AppData\Local\Python\pyth

In [ ]:
with open("texto.text", "w") as arquivo:
    arquivo.write(f"{doc.document}")

TypeError: write() argument must be str, not DoclingDocument

### Forma 1 de Criar o DataFrame para passar a IA

In [ ]:
import pandas as pd
tb = pd.DataFrame()
for idx, table in enumerate(doc.tables):
    df = table.export_to_dataframe()
    tb = pd.concat([tb, df], ignore_index=True)

2026-01-14 07:18:50,469 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 07:18:50,470 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 07:18:50,474 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 07:18:50,475 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 07:18:50,478 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 07:18:50,480 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


In [ ]:
aprovados = pd.DataFrame(columns=tb.columns)
for linha in tb.values:
    if linha[7] in ('Aprovado', 'Ap', 'Dispensado'):
        aprovados = pd.concat([aprovados, pd.DataFrame([linha])], ignore_index=True)




### Forma 2 de Criar o DataFrame para passar a IA

In [ ]:
import unicodedata
import re
from ftfy import fix_text

def normalizar_coluna(col):
    col = col.strip()
    col = unicodedata.normalize('NFKD', col)
    col = ''.join(c for c in col if not unicodedata.combining(c))
    col = col.lower()
    col = re.sub(r'\s+', '_', col)
    col = re.sub(r'[^a-z0-9_]', '', col)
    return col




In [120]:
dfs = []

for table in doc.tables:
    dfs.append(table.export_to_dataframe())

tb = pd.concat(dfs, ignore_index=True)

tb = tb.drop(columns=[0, 1], errors='ignore')

tb.columns = [
    normalizar_coluna(c) if i not in (0, 1) else c
    for i, c in enumerate(tb.columns)
]

for col in tb.select_dtypes(include='object').columns:
    tb[col] = tb[col].apply(limpar_texto)

aprovados = tb[tb['situacao'].isin(['Aprovado', 'Ap', 'Dispensado'])]

2026-01-14 09:12:51,692 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:51,699 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:51,706 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:51,708 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:51,711 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:51,712 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


In [100]:
import json

dados = json.loads(aprovados.to_json(orient='index'))
json_final = json.dumps(dados, ensure_ascii=False)


In [101]:
json_final

'{"3": {"Semestre": "2023.1", "Código": "S4-B0297", "disciplina": "CORPOREIDADE E MOTRICIDADE HUMANA", "ch": "30.00", "mdia": "8.65", "situacao": "Aprovado"}, "6": {"Semestre": "2023.1", "Código": "S4-B0342", "disciplina": "EVOLU˙ˆO HISTÓRICA DA FISIOTERAPIA", "ch": "90.00", "mdia": "7.38", "situacao": "Aprovado"}, "7": {"Semestre": "2023.1", "Código": "S4-B0368", "disciplina": "FUNDAMENTOS DE A˙ÕES PREVENTIVAS EM SAÚDE", "ch": "30.00", "mdia": "5.50", "situacao": "Aprovado"}, "8": {"Semestre": "2023.1", "Código": "S4-B0369", "disciplina": "FUNDAMENTOS DE SAÚDE COLETIVA", "ch": "30.00", "mdia": "8.50", "situacao": "Aprovado"}, "11": {"Semestre": "2023.1", "Código": "S4-B0404", "disciplina": "PRIMEIROS SOCORROS", "ch": "30.00", "mdia": "8.65", "situacao": "Aprovado"}, "12": {"Semestre": "2023.1", "Código": "S4-B0415", "disciplina": "PSICOLOGIA APLICADA À FISIOTERAPIA", "ch": "60.00", "mdia": "10.00", "situacao": "Aprovado"}, "13": {"Semestre": "2023.1", "Código": "S4-B0424", "disciplina

## JSON mais facil para IA


In [121]:
dfs = []

for table in doc.tables:
    dfs.append(table.export_to_dataframe())

tb = pd.concat(dfs, ignore_index=True)

tb = tb.drop(columns=[0, 1], errors='ignore')

tb.columns = [
    normalizar_coluna(c) if i not in (0, 1) else c
    for i, c in enumerate(tb.columns)
]

for col in tb.select_dtypes(include='object').columns:
    tb[col] = tb[col].apply(limpar_texto)

aprovados = tb[tb['situacao'].isin(['Aprovado', 'Ap', 'Dispensado'])]

2026-01-14 09:12:59,108 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:59,109 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:59,113 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:59,114 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:59,118 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2026-01-14 09:12:59,120 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


In [122]:
def chave_disciplina(s):
    if not isinstance(s, str):
        return "DESCONHECIDA"
    return (
        s.strip()
         .upper()
         .replace("  ", " ")
    )


In [125]:
json_aprovados = {}

for _, row in aprovados.iterrows():
    nome = chave_disciplina(row["disciplina"])

    json_aprovados[nome] = {
        "semestre": row["Semestre"],
        "codigo": row["Código"],
        "carga_horaria": row["ch"],
        "media": row["mdia"],
        "situacao": row["situacao"],
    }


In [ ]:
json

{'CORPOREIDADE E MOTRICIDADE HUMANA': {'semestre': '2023.1',
  'codigo': 'S4-B0297',
  'carga_horaria': '30.00',
  'media': '8.65',
  'situacao': 'Aprovado'},
 'EVOLU˙ˆO HISTÓRICA DA FISIOTERAPIA': {'semestre': '2023.1',
  'codigo': 'S4-B0342',
  'carga_horaria': '90.00',
  'media': '7.38',
  'situacao': 'Aprovado'},
 'FUNDAMENTOS DE A˙ÕES PREVENTIVAS EM SAÚDE': {'semestre': '2023.1',
  'codigo': 'S4-B0368',
  'carga_horaria': '30.00',
  'media': '5.50',
  'situacao': 'Aprovado'},
 'FUNDAMENTOS DE SAÚDE COLETIVA': {'semestre': '2023.1',
  'codigo': 'S4-B0369',
  'carga_horaria': '30.00',
  'media': '8.50',
  'situacao': 'Aprovado'},
 'PRIMEIROS SOCORROS': {'semestre': '2023.1',
  'codigo': 'S4-B0404',
  'carga_horaria': '30.00',
  'media': '8.65',
  'situacao': 'Aprovado'},
 'PSICOLOGIA APLICADA À FISIOTERAPIA': {'semestre': '2023.1',
  'codigo': 'S4-B0415',
  'carga_horaria': '60.00',
  'media': '10.00',
  'situacao': 'Aprovado'},
 'TÉCNICAS DE INFORM`TICA': {'semestre': '2023.1',
  '